## ⚠️ Important: Colab Extension Limitation

**The VS Code Colab extension runs on Google's servers, not your local machine.**

Your local files aren't automatically available. You have two options:

**Option A (Recommended):** Train locally on your Mac
- Close this notebook
- Open terminal in VS Code
- Run: `python src/train.py --zones IT-NORD --epochs 50 --attention`
- Slower (~16 hours) but uses your local files

**Option B:** Upload to Google Colab website
1. Go to https://colab.research.google.com
2. Upload this notebook
3. Upload your data folder manually
4. Run there (faster, 2-3 hours with GPU)

**If you want to continue here**, run the cells below to upload your files to Colab.

# Train Attention Model on Google Colab GPU

This notebook trains the attention-enhanced CNN-LSTM model using Google Colab's free GPU.

**Expected Speed:**
- Mac MPS: ~20 min/epoch → 16+ hours total
- Colab T4 GPU: ~2-3 min/epoch → 2-3 hours total ⚡

**How to use:**
1. Click on cells one by one and press **Shift+Enter** to run them
2. Or use **Run All** button at the top
3. Wait for each cell to complete before moving to the next (watch for ✓ checkmark)

## 1. Check GPU is Available ✅

First, let's verify that we're connected to a GPU (not CPU)

In [1]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Count: {torch.cuda.device_count()}")
    print("\n🚀 Ready to train on GPU!")
else:
    print("\n⚠️ WARNING: No GPU detected!")
    print("Training will be very slow on CPU.")
    print("Make sure you selected a Colab kernel.")

PyTorch version: 2.8.0+cu126
CUDA available: False

⚠️ WARNING: No GPU detected!
Training will be very slow on CPU.
Make sure you selected a Colab kernel.


## 2. Verify Project Files 📁

Check that all data files are present

In [2]:
import os

print(f"Current directory: {os.getcwd()}")
print(f"\nListing current directory contents:")
print(os.listdir('.'))

# Check if we need to navigate into the repo folder
if os.path.exists('ACIT4620-exam'):
    print("\n✓ Found ACIT4620-exam folder")
    os.chdir('ACIT4620-exam')
    print(f"Changed to: {os.getcwd()}")
    print(f"\nContents: {os.listdir('.')}")

# Now check for data
if os.path.exists('data/processed/train'):
    train_files = os.listdir('data/processed/train')
    test_files = os.listdir('data/processed/test')
    print(f"\n✓ Training files: {len(train_files)} zones")
    print(f"✓ Test files: {len(test_files)} zones")
else:
    print("\n⚠️ data/processed/train not found!")
    print("Available directories:", [d for d in os.listdir('.') if os.path.isdir(d)])

# Check source code
if os.path.exists('src/train.py') and os.path.exists('src/model_attention.py'):
    print(f"✓ Source code ready")
else:
    print(f"⚠️ Missing source files!")

Current directory: /content

Checking data files...


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/train'

## 3. Train Attention Model 🚀

This will take approximately **2-3 hours** on Colab T4 GPU.

The training will:
- Use 19 features (13 weather + 2 engineered + 4 temporal)
- Train attention-enhanced CNN-LSTM model (275K parameters)
- Save best model automatically to `models/it-nord/`

In [ ]:
# Train with attention mechanism
# Press Shift+Enter and wait for completion (progress bars will show)
!python src/train.py --zones IT-NORD --epochs 50 --attention

## 4. View Training Results 📊

Let's see how the training went

In [ ]:
# View training history
import json
import pandas as pd

with open('models/it-nord/training_history.json', 'r') as f:
    history = json.load(f)

# Convert to DataFrame for better display
df = pd.DataFrame(history)
df['epoch'] = range(1, len(df) + 1)
df = df[['epoch', 'train_loss', 'train_mae', 'val_loss', 'val_mae', 'lr']]

print("📊 Training History:")
print(df.to_string(index=False))

# Find best epoch
best_epoch = df['val_loss'].idxmin() + 1
print(f"\n🏆 Best Epoch: {best_epoch}")
print(f"   Validation Loss: {df.loc[best_epoch-1, 'val_loss']:.5f}")
print(f"   Validation MAE: {df.loc[best_epoch-1, 'val_mae']:.5f}")

In [ ]:
# Display training curves
from IPython.display import Image
Image('models/it-nord/training_curves.png')

## 5. Evaluate Model Performance 🎯

Test the model on real data with both forecast and actual weather

In [ ]:
# Evaluate with dual-scenario (forecast and actual weather)
!python src/evaluate_forecast.py --zones IT-NORD --model-dir models/it-nord --dual-scenario

In [ ]:
# View detailed results
import pandas as pd

metrics = pd.read_csv('results/it-nord/metrics_comparison.csv')
print("\n📈 Model Performance Metrics:")
print(metrics.to_string(index=False))

# Compare with baseline
print("\n" + "="*60)
print("📊 COMPARISON WITH BASELINE:")
print("="*60)
print("Baseline (no temporal features):  MAE = 515 MW")
print("Temporal Features (base model):   MAE = 409 MW (-106 MW, -20.6%)")
print(f"Attention Model (this run):        MAE = {metrics.loc[0, 'Forecast Weather'].split()[0]} MW")
print("="*60)

## ✅ Done!

Your attention model is now trained and saved in `models/it-nord/`.

**Next steps:**
- The model has been automatically saved locally
- You can compare results with the base model (MAE=409 MW)
- If attention model is better, keep it; otherwise revert to the base model

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Display training curves
print("📈 Training Curves:")
display(Image('models/it-nord/training_curves.png'))

# Display forecast comparison
print("\n📊 Forecast vs Actual:")
display(Image('results/it-nord/forecast_comparison.png'))

# Display scatter plot
print("\n🎯 Prediction Accuracy:")
display(Image('results/it-nord/scatter_comparison.png'))

## 6. View Visualizations 📉

Display the prediction plots